In [1]:
%cd D:\Study Documents\Other subjects\CoDraft-Bench
CD_PATH = "/kaggle/working/CoDraft-Bench"

D:\Study Documents\Other subjects\CoDraft-Bench


In [2]:
import os
import sys
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

if CD_PATH not in sys.path:
    sys.path.append(CD_PATH)

In [4]:
from copy import deepcopy
import numpy as np
import pandas as pd
import torch

from config import *
from preprocess import *
from model import *
from model.models import *
from utils import *

In [5]:
RANDOM_SEED = 42
set_seed(RANDOM_SEED)

data_generator = torch.Generator()
data_generator.manual_seed(RANDOM_SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

Random seed set to 42
Device: cpu


In [ ]:
INPUT_ROOT = "dataset"
WORK_DIR = "working"
MODEL_NAME = "microsoft/deberta-v3-base"
MODEL_TYPE = "cross-encoder"
CHECKPOINT_FILE = f"{WORK_DIR}/{MODEL_NAME}_best_model.pth"
PRETRAIN_FILE = None

In [7]:
tokenizer = get_tokenizer(MODEL_NAME)

In [8]:
data_manager = DataManager(
    input_root = INPUT_ROOT, 
    work_dir = WORK_DIR, 
    config_data = CONFIG_DATA,
    tokenizer=tokenizer,
    seed_worker = seed_worker,
    data_generator = data_generator,
    random_seed = RANDOM_SEED,
    rebalance=True
)

Number of new cross_pairing: 134124


Map: 100%|██████████| 8534/8534 [00:04<00:00, 2078.71 examples/s]


In [9]:
train_dataloader, evaluator = data_manager.get_dataloaders(model_type="cross_encoder")

In [10]:
train_df, val_df, test_df = data_manager.get_data()

In [16]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class_weights = compute_class_weight(train_df['label_score'])
weights_tensor = torch.tensor(class_weights, dtype=torch.float32).to(device)

In [17]:
model, _ = get_model(model_type="cross_encoder",model_name=MODEL_NAME, num_classes=5, max_len=256, weights_tensor=weights_tensor)

Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at microsoft/deberta-v3-base and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
trained_model = train_cross_encoder(model, train_dataloader, evaluator)

Epoch:   0%|          | 0/5 [00:47<?, ?it/s]


KeyboardInterrupt: 

In [ ]:
test_preds, test_true = get_preds_cross_encoder(model=trained_model, df_test=test_df)

In [ ]:
result_df = pd.DataFrame({
    "label": test_true,
    "pred": test_preds,
})

In [ ]:
result_df.to_csv("result_df.csv", index=False)

In [ ]:
save_model(trained_model, tokenizer, MODEL_NAME, './model/checkpoint')

In [ ]:
zip_model_folder("./model")